In [ ]:
%pip install torchmetrics

In [33]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.utils.data as data_utils


import pandas as pd

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split

import numpy as np
import random

from sklearn.datasets import load_iris

# StandardScaler

from sklearn.preprocessing import StandardScaler
from torchmetrics import Accuracy

%matplotlib inline

In [3]:
iris = load_iris()
X = iris.data
y = iris.target
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [8]:
X_train.shape, y_train.shape, X_test.shape, y_test.shape

((120, 4), (120,), (30, 4), (30,))

In [11]:
classes = np.unique(y_train)

In [15]:
ss = StandardScaler()

X_train = ss.fit_transform(X_train)
X_test = ss.transform(X_test)

X_train.shape, y_train.shape, X_test.shape, y_test.shape

((120, 4), (120,), (30, 4), (30,))

In [16]:
np.mean(X_train, axis=0), np.std(X_train, axis=0), np.min(X_train, axis=0), np.max(X_train, axis=0)

(array([ 1.71344420e-15, -1.66579713e-15, -2.23894977e-16, -5.73615229e-17]),
 array([1., 1., 1., 1.]),
 array([-1.83962751, -2.37377751, -1.56253475, -1.44608785]),
 array([2.30486738, 2.99237573, 1.70388875, 1.75755292]))

In [ ]:
# Convert to PyTorch tensors

X_train = torch.tensor(X_train, dtype=torch.float64)
y_train = torch.tensor(y_train, dtype=torch.long)
X_test = torch.tensor(X_test, dtype=torch.float64)
y_test = torch.tensor(y_test, dtype=torch.long)

In [22]:
# Convert to DataLoader

train_ds = data_utils.TensorDataset(X_train, y_train)
test_ds = data_utils.TensorDataset(X_test, y_test)

In [23]:
train_dl = data_utils.DataLoader(train_ds, batch_size=16, shuffle=True)
test_dl = data_utils.DataLoader(test_ds, batch_size=16, shuffle=False)

In [24]:
for i, data in enumerate(train_dl):
    print(f"Data: {data[0]}, Label: {data[1]}, Index: {i}")

Data: tensor([[-0.9863, -2.3738, -0.1299, -0.2447],
        [ 0.2326, -1.9266,  0.1566, -0.2447],
        [-0.2550, -0.1379,  0.2139,  0.1557],
        [ 0.5983,  0.5329,  1.3027,  1.7576],
        [ 1.0859, -0.1379,  0.7297,  0.6897],
        [-1.4739,  0.7565, -1.3333, -1.1791],
        [-0.3769, -1.0322,  0.3859,  0.0222],
        [-0.7426,  2.3216, -1.2760, -1.4461],
        [-0.8645,  0.5329, -1.1614, -0.9121],
        [ 1.5735, -0.1379,  1.2454,  1.2236],
        [ 0.7202,  0.3093,  0.4432,  0.4227],
        [-1.2301,  0.7565, -1.2187, -1.3126],
        [ 1.0859,  0.0857,  1.0735,  1.6241],
        [-0.8645,  0.7565, -1.2760, -1.3126],
        [ 0.1107,  0.3093,  0.6151,  0.8232],
        [-1.7177,  0.3093, -1.3906, -1.3126]], dtype=torch.float64), Label: tensor([1, 1, 1, 2, 1, 0, 1, 0, 0, 2, 1, 0, 2, 0, 1, 0]), Index: 0
Data: tensor([[-0.9863,  0.7565, -1.2760, -1.3126],
        [-0.7426,  0.9801, -1.2760, -1.3126],
        [ 1.0859,  0.5329,  1.1308,  1.7576],
        [-1.4739,

In [25]:
class MLP(nn.Module):
    
    def __init__(self):
        super().__init__()

        self.layer_1 = nn.Linear(in_features=4, out_features=32)
        self.layer_2 = nn.Linear(in_features=32, out_features=3)

    def forward(self, x):
        x = self.layer_1(x)
        x = F.relu(x)
        x = self.layer_2(x)
        x = F.softmax(x, dim=1)

        return x

In [26]:
model = MLP()

In [30]:
for i in model.parameters():
    print(i.shape, i.dtype, i.device)

torch.Size([32, 4]) torch.float32 cpu
torch.Size([32]) torch.float32 cpu
torch.Size([3, 32]) torch.float32 cpu
torch.Size([3]) torch.float32 cpu


In [35]:


loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(params=model.parameters(), lr=0.01)

#Accuracy
accuracy_fn = Accuracy(task="multiclass", num_classes=classes.size, average="macro")


In [39]:
epochs = 50

train_loss_values = []
test_loss_values = []
epoch_count = []
accuracy_values = []

for epoch in range(epochs):
    model.train()
    loss_epoch = 0
    accuracy_epoch = 0

    for i, data in enumerate(train_dl, 0):

      X = data[0].float()
      y = data[1]

      y_pred = model(X) 
      loss = loss_fn(y_pred, y)
      accuracy = accuracy_fn(y_pred, y)

      accuracy_epoch += accuracy 
      loss_epoch += loss

      # 3. Azzeramento dei gradienti
      optimizer.zero_grad()

      # 4. Backpropagation
      loss.backward()

      # 5. Ottimizzazione
      optimizer.step()

    loss_test = 0
    model.eval()
    for j, data in enumerate(test_dl, 0):

      X = data[0] #[128, 8]
      y = data[1] #[128 ,1]

      accuracy_epoch = 0

      with torch.no_grad():

        # 1. Forward pass
        y_pred = model(X) #[128 ,1]
        # 2. Calculo della loss
        loss = loss_fn(y_pred, y)
        accuracy = accuracy_fn(y_pred, y)

        accuracy_epoch += accuracy 

      loss_test += loss

    epoch_count.append(epoch)
    train_loss_values.append(loss_epoch.detach().numpy()/len(train_dl))
    test_loss_values.append(loss_test.detach().numpy()/len(test_dl))

    accuracy_values.append(accuracy_epoch.detach().numpy()/len(test_dl))

    print(f"Epoca: {epoch} |  Train Loss: {loss_epoch/len(train_dl):.3f} | Test Loss: {loss_test/len(test_dl):.3f} | Accuracy: {accuracy_epoch/len(test_dl):.3f}")

RuntimeError: mat1 and mat2 must have the same dtype, but got Double and Float